### Machine Learning Pipeline

The dataset used for this example is obtained from [UC Irvine Machine Learning Repository](https://archive.ics.uci.edu/dataset/222/bank+marketing).

The second part consists of creating a machine learning model to predict if the client will subscribe (yes/no) a term deposit (variable y).

#### Import Dataset

In [16]:
# import libraries
import pandas as pd
import joblib
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import f1_score

In [23]:
# read dataset
df = pd.read_csv("datasets/bank-full_train_test.csv")

In [17]:
df.head()

,age,default,balance,housing,loan,day,month,duration,campaign,pdays,...,job_unemployed,job_unknown,education_secondary,education_tertiary,education_unknown,poutcome_other,poutcome_success,poutcome_unknown,contact_telephone,contact_unknown
0,43,0,50,0,0,28,5,90,8,-1,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,1.0
1,32,0,40,0,0,8,8,419,2,-1,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0
2,32,0,2559,0,0,7,7,889,1,-1,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
3,23,0,480,0,0,9,2,742,2,182,...,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,32,0,1169,1,0,21,5,322,2,-1,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,1.0


In [18]:
df.describe()

,age,default,balance,housing,loan,day,month,duration,campaign,pdays,...,job_unemployed,job_unknown,education_secondary,education_tertiary,education_unknown,poutcome_other,poutcome_success,poutcome_unknown,contact_telephone,contact_unknown
count,9470.000000,9470.000000,9470.000000,9470.000000,9470.000000,9470.000000,9470.000000,9470.000000,9470.000000,9470.000000,...,9470.000000,9470.000000,9470.000000,9470.000000,9470.000000,9470.000000,9470.000000,9470.000000,9470.000000,9470.000000
mean,41.385428,0.013094,1565.743083,0.470011,0.131679,15.574762,6.223970,378.334213,2.495248,52.460401,...,0.032101,0.006653,0.491130,0.329567,0.042872,0.049525,0.099366,0.739704,0.067582,0.207181
std,11.995102,0.113683,3209.764232,0.499126,0.338159,8.389508,2.576857,349.370763,2.678249,109.175676,...,0.176279,0.081296,0.499948,0.470081,0.202580,0.216973,0.299169,0.438819,0.251040,0.405307
min,18.000000,0.000000,-6847.000000,0.000000,0.000000,1.000000,1.000000,5.000000,1.000000,-1.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,32.000000,0.000000,125.000000,0.000000,0.000000,8.000000,5.000000,143.000000,1.000000,-1.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,39.000000,0.000000,569.500000,0.000000,0.000000,15.000000,6.000000,260.000000,2.000000,-1.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000
75%,49.000000,0.000000,1775.750000,1.000000,0.000000,21.000000,8.000000,506.750000,3.000000,66.500000,...,0.000000,0.000000,1.000000,1.000000,0.000000,0.000000,0.000000,1.000000,0.000000,0.000000
max,95.000000,1.000000,81204.000000,1.000000,1.000000,31.000000,12.000000,3881.000000,43.000000,854.000000,...,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


In [22]:
df

,age,default,balance,housing,loan,day,month,duration,campaign,pdays,...,job_unemployed,job_unknown,education_secondary,education_tertiary,education_unknown,poutcome_other,poutcome_success,poutcome_unknown,contact_telephone,contact_unknown


In [27]:
df["y"].value_counts()

y
no     4735
yes    4735
Name: count, dtype: int64

#### Transform Dataset

In [25]:
# balance dataset
def balance_dataset(df: pd.DataFrame) -> pd.DataFrame:
    # Separate the classes
    df_y0 = df[df['y'] == 'no']
    df_y1 = df[df['y'] == 'yes']

    # Find the smaller class size
    min_size = len(df_y1)

    # Randomly sample from each class
    df_y0_balanced = df_y0.sample(n=min_size, random_state=42)

    # Concatenate back together
    df_balanced = pd.concat([df_y0_balanced, df_y1])

    # Shuffle the dataset
    df_balanced = df_balanced.sample(frac=1, random_state=42).reset_index(drop=True)

    return df_balanced

df = balance_dataset(df)

In [26]:
df

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,43,blue-collar,married,primary,no,50,no,no,unknown,28,may,90,8,-1,0,unknown,no
1,32,technician,single,secondary,no,40,no,no,cellular,8,aug,419,2,-1,0,unknown,yes
2,32,self-employed,single,tertiary,no,2559,no,no,cellular,7,jul,889,1,-1,0,unknown,yes
3,23,student,single,secondary,no,480,no,no,cellular,9,feb,742,2,182,1,failure,yes
4,32,management,single,tertiary,no,1169,yes,no,unknown,21,may,322,2,-1,0,unknown,no
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9465,27,blue-collar,married,tertiary,no,335,yes,no,unknown,26,jun,519,3,-1,0,unknown,yes
9466,60,management,married,tertiary,no,315,no,no,cellular,1,sep,840,1,183,3,success,yes
9467,34,technician,single,tertiary,no,2838,yes,no,cellular,13,may,620,3,-1,0,unknown,yes
9468,35,technician,married,tertiary,no,1473,yes,no,unknown,12,may,84,3,-1,0,unknown,no


In [28]:
# map target
df.loc[df["y"] == "yes", "y"] = 1
df.loc[df["y"] == "no", "y"] = 0
df["y"] = df["y"].astype(int)

In [29]:
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,43,blue-collar,married,primary,no,50,no,no,unknown,28,may,90,8,-1,0,unknown,0
1,32,technician,single,secondary,no,40,no,no,cellular,8,aug,419,2,-1,0,unknown,1
2,32,self-employed,single,tertiary,no,2559,no,no,cellular,7,jul,889,1,-1,0,unknown,1
3,23,student,single,secondary,no,480,no,no,cellular,9,feb,742,2,182,1,failure,1
4,32,management,single,tertiary,no,1169,yes,no,unknown,21,may,322,2,-1,0,unknown,0


In [5]:
# prepare dataset for classification model
class Transformer:
    def __init__(self):
        self.DROP_COLUMNS = []
        self.BINARY_FEATURES = [
            "housing",
            "loan",
            "default",
        ]

    def transform(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.drop(self.DROP_COLUMNS, axis=1)
        df = self._map_binary_column_to_int(df)
        df = self._map_month_to_int(df)

        return df

    def _map_binary_column_to_int(self, df: pd.DataFrame) -> pd.DataFrame:
        for col in self.BINARY_FEATURES:
            df[col] = df[col].map({"yes": 1, "no": 0})
        return df

    def _map_month_to_int(self, df: pd.DataFrame) -> pd.DataFrame:
        month_mapping = {
            "jan": 1,
            "feb": 2,
            "mar": 3,
            "apr": 4,
            "may": 5,
            "jun": 6,
            "jul": 7,
            "aug": 8,
            "sep": 9,
            "oct": 10,
            "nov": 11,
            "dec": 12,
        }
        df["month"] = df["month"].map(month_mapping)

        return df

In [30]:
# transform the dataset
df = Transformer().transform(df)

In [31]:
df.head()

,age,job,marital,education,default,balance,housing,loan,contact,day,month,duration,campaign,pdays,previous,poutcome,y
0,43,blue-collar,married,primary,0,50,0,0,unknown,28,5,90,8,-1,0,unknown,0
1,32,technician,single,secondary,0,40,0,0,cellular,8,8,419,2,-1,0,unknown,1
2,32,self-employed,single,tertiary,0,2559,0,0,cellular,7,7,889,1,-1,0,unknown,1
3,23,student,single,secondary,0,480,0,0,cellular,9,2,742,2,182,1,failure,1
4,32,management,single,tertiary,0,1169,1,0,unknown,21,5,322,2,-1,0,unknown,0


In [32]:
# encode categorical variables
ONE_HOT_ENCODE_COLUMNS = [
            "marital",
            "job",
            "education",
            "poutcome",
            "contact",
        ]

encoder = OneHotEncoder(drop='first', sparse_output=False).set_output(transform="pandas")
encoder.fit(df[ONE_HOT_ENCODE_COLUMNS])
encoded_df = encoder.transform(df[ONE_HOT_ENCODE_COLUMNS])
df = df.drop(columns=ONE_HOT_ENCODE_COLUMNS)
df = pd.concat([df, encoded_df], axis=1)

In [34]:
df.iloc[0]

age                    43.0
default                 0.0
balance                50.0
housing                 0.0
loan                    0.0
day                    28.0
month                   5.0
duration               90.0
campaign                8.0
pdays                  -1.0
previous                0.0
y                       0.0
marital_married         1.0
marital_single          0.0
job_blue-collar         1.0
job_entrepreneur        0.0
job_housemaid           0.0
job_management          0.0
job_retired             0.0
job_self-employed       0.0
job_services            0.0
job_student             0.0
job_technician          0.0
job_unemployed          0.0
job_unknown             0.0
education_secondary     0.0
education_tertiary      0.0
education_unknown       0.0
poutcome_other          0.0
poutcome_success        0.0
poutcome_unknown        1.0
contact_telephone       0.0
contact_unknown         1.0
Name: 0, dtype: float64

In [35]:
# save encoder - future use
joblib.dump(encoder, 'model_utils/encoder.pkl')

['model_utils/encoder.pkl']

#### Train and evaluate model

In [36]:
# Split the data into training and test sets
X = df.drop('y', axis=1)
y = df['y']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [10]:
# Define the model hyperparameters
params = {
    "solver": "lbfgs",
    "max_iter": 1000,
    "multi_class": "auto",
    "random_state": 8888,
}

In [37]:
# Train the model
lr = LogisticRegression(**params)
lr.fit(X_train, y_train)

c:\Users\Robert\Documents\EADA\2026_MLOPS_SD\mlops-and-system-design\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(
c:\Users\Robert\Documents\EADA\2026_MLOPS_SD\mlops-and-system-design\.venv\Lib\site-packages\sklearn\linear_model\_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


LogisticRegression(max_iter=1000, multi_class='auto', random_state=8888)

In [38]:
# Predict on the test set
y_pred = lr.predict(X_test)

In [39]:
# Calculate metrics
f1_score = f1_score(y_test, y_pred)

### MLFlow Integration

In [40]:
# import libraries
import mlflow
from mlflow.models import infer_signature

In a new terminal, run a local Tracking Server with the following command:

`mlflow server --host 127.0.0.1 --port 8080`

In [41]:
# Set our tracking server uri for logging
mlflow.set_tracking_uri(uri="http://127.0.0.1:8080")

In [42]:
params

{'solver': 'lbfgs',
 'max_iter': 1000,
 'multi_class': 'auto',
 'random_state': 8888}

In [43]:
# Create a new MLflow Experiment
mlflow.set_experiment("LR Experiment")

# Start an MLflow run
with mlflow.start_run():
    # Log the hyperparameters
    mlflow.log_params(params)

    # Log the loss metric
    mlflow.log_metric("f1_score", f1_score)

    # Set a tag that we can use to remind ourselves what this run was for
    mlflow.set_tag("Training Info", "Basic LR model to present MLFlow capabilities")

    # Infer the model signature
    signature = infer_signature(X_train, lr.predict(X_train))

    # Log the model
    model_info = mlflow.sklearn.log_model(
        sk_model=lr,
        artifact_path="bank_model",
        signature=signature,
        input_example=X_train,
        registered_model_name="bank-model-basic",
    )

2026/05/14 19:02:14 INFO mlflow.tracking.fluent: Experiment with name 'LR Experiment' does not exist. Creating a new experiment.
c:\Users\Robert\Documents\EADA\2026_MLOPS_SD\mlops-and-system-design\.venv\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
Successfully registered model 'bank-model-basic'.
2026/05/14 19:02

🏃 View run abrasive-bat-472 at: http://127.0.0.1:8080/#/experiments/365175901962808741/runs/d26ebcab71394ea5b3861d72f9f6bd86
🧪 View experiment at: http://127.0.0.1:8080/#/experiments/365175901962808741


Created version '1' of model 'bank-model-basic'.
